In [0]:
from pyspark.sql import functions as F

catalog = "workspace"
schema = "de_ecommerce"

def bronze(name):
    return spark.table(f"{catalog}.{schema}.bronze_{name}")

def save_silver(df, name, rows_before):
    df.write.format("delta").mode("overwrite").saveAsTable(f"{catalog}.{schema}.silver_{name}")
    rows_after = spark.table(f"{catalog}.{schema}.silver_{name}").count()
    print(f"{name}: {rows_before} -> {rows_after} rows ({rows_before - rows_after} removed)")

def clean_zip(col):
    return F.lpad(F.col(col).cast("string"), 5, "0")

meta = ["source_file", "ingested_at"]

In [0]:
# ---- orders ----
df = bronze("orders").drop(*meta)
n = df.count()
for c in ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
          "order_delivered_customer_date", "order_estimated_delivery_date"]:
    df = df.withColumn(c, F.col(c).cast("timestamp"))
df = (df.dropDuplicates(["order_id"])
        .filter(F.col("order_id").isNotNull() & F.col("customer_id").isNotNull())
        .withColumn("order_status", F.lower(F.trim("order_status")))
        .filter(F.col("order_delivered_customer_date").isNull() |
                (F.col("order_delivered_customer_date") >= F.col("order_purchase_timestamp"))))
save_silver(df, "orders", n)

# ---- customers ----
df = bronze("customers").drop(*meta)
n = df.count()
df = (df.dropDuplicates(["customer_id"])
        .filter(F.col("customer_id").isNotNull())
        .withColumn("customer_zip_code_prefix", clean_zip("customer_zip_code_prefix"))
        .withColumn("customer_city", F.lower(F.trim("customer_city")))
        .withColumn("customer_state", F.upper(F.trim("customer_state"))))
save_silver(df, "customers", n)

# ---- sellers ----
df = bronze("sellers").drop(*meta)
n = df.count()
df = (df.dropDuplicates(["seller_id"])
        .filter(F.col("seller_id").isNotNull())
        .withColumn("seller_zip_code_prefix", clean_zip("seller_zip_code_prefix"))
        .withColumn("seller_city", F.lower(F.trim("seller_city")))
        .withColumn("seller_state", F.upper(F.trim("seller_state"))))
save_silver(df, "sellers", n)

orders: 99441 -> 99441 rows (0 removed)
customers: 99441 -> 99441 rows (0 removed)
sellers: 3095 -> 3095 rows (0 removed)


In [0]:
df = bronze("products").drop(*meta)
n = df.count()
trans = bronze("category_translation").select("product_category_name", "product_category_name_english")

df = (df.dropDuplicates(["product_id"])
        .filter(F.col("product_id").isNotNull())
        .withColumnRenamed("product_name_lenght", "product_name_length")
        .withColumnRenamed("product_description_lenght", "product_description_length")
        .join(trans, "product_category_name", "left")
        .withColumn("product_category_name_english",
                    F.coalesce(F.col("product_category_name_english"),
                               F.col("product_category_name"),
                               F.lit("unknown"))))
save_silver(df, "products", n)

products: 32951 -> 32951 rows (0 removed)


In [0]:
# ---- order_items ----
df = bronze("order_items").drop(*meta)
n = df.count()
df = (df.dropDuplicates(["order_id", "order_item_id"])
        .filter(F.col("order_id").isNotNull() & F.col("product_id").isNotNull() & F.col("seller_id").isNotNull())
        .filter((F.col("price") > 0) & (F.col("freight_value") >= 0))
        .withColumn("shipping_limit_date", F.col("shipping_limit_date").cast("timestamp")))
save_silver(df, "order_items", n)

# ---- order_payments ----
df = bronze("order_payments").drop(*meta)
n = df.count()
df = (df.dropDuplicates(["order_id", "payment_sequential"])
        .filter(F.col("order_id").isNotNull())
        .filter(F.col("payment_value") >= 0)
        .filter(F.col("payment_type") != "not_defined"))
save_silver(df, "order_payments", n)

# ---- order_reviews ----
df = bronze("order_reviews").drop(*meta)
n = df.count()
df = (df.withColumn("review_score", F.col("review_score").cast("int"))
        .withColumn("review_creation_date", F.col("review_creation_date").cast("timestamp"))
        .withColumn("review_answer_timestamp", F.col("review_answer_timestamp").cast("timestamp"))
        .filter(F.col("order_id").isNotNull() & F.col("review_score").between(1, 5))
        .dropDuplicates(["review_id", "order_id"]))
save_silver(df, "order_reviews", n)

order_items: 112650 -> 112650 rows (0 removed)
order_payments: 103886 -> 103883 rows (3 removed)
order_reviews: 99224 -> 99224 rows (0 removed)


In [0]:
df = bronze("geolocation").drop(*meta)
n = df.count()
df = (df.filter(F.col("geolocation_lat").between(-34, 6) & F.col("geolocation_lng").between(-74, -34))
        .withColumn("zip_code_prefix", clean_zip("geolocation_zip_code_prefix"))
        .groupBy("zip_code_prefix")
        .agg(F.avg("geolocation_lat").alias("lat"),
             F.avg("geolocation_lng").alias("lng"),
             F.first(F.lower(F.trim("geolocation_city"))).alias("city"),
             F.first(F.upper(F.trim("geolocation_state"))).alias("state")))
save_silver(df, "geolocation", n)

geolocation: 1000163 -> 19010 rows (981153 removed)


In [0]:
display(spark.sql(f"SHOW TABLES IN {catalog}.{schema}"))

database,tableName,isTemporary
de_ecommerce,bronze_category_translation,false
de_ecommerce,bronze_customers,false
de_ecommerce,bronze_geolocation,false
de_ecommerce,bronze_order_items,false
de_ecommerce,bronze_order_payments,false
de_ecommerce,bronze_order_reviews,false
de_ecommerce,bronze_orders,false
de_ecommerce,bronze_products,false
de_ecommerce,bronze_sellers,false
de_ecommerce,silver_customers,false
